In [0]:
import json
import os
import time
from datetime import UTC, datetime

import requests

url = "https://overpass-api.de/api/interpreter"
query = """
[out:json][timeout:120];
area["ISO3166-1"="MA"][admin_level=2]->.morocco;
nwr["amenity"="bank"](area.morocco);
out tags center;
"""

dbutils.widgets.text("ingest_date", datetime.now(UTC).strftime("%Y-%m-%d"))
ingest_date = dbutils.widgets.get("ingest_date")
target_dir = f"/Volumes/workspace/bank_lakehouse/landing/osm_banks/ingest_date={ingest_date}"
target_path = f"{target_dir}/banks.json"


def fetch_with_backoff(max_attempts=5, first_wait=10):
    wait = first_wait
    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.post(
                url,
                data={"data": query},
                headers={"User-Agent": "bank-reviews-lakehouse-student-project"},
                timeout=180,
            )
            print(f"Tentative {attempt} : statut {response.status_code}")
            if response.status_code == 200:
                return response.text
            if response.status_code not in (429, 502, 503, 504):
                raise RuntimeError(f"Erreur non temporaire : {response.text[:300]}")
        except requests.RequestException as error:
            print(f"Tentative {attempt} : erreur {type(error).__name__}")

        if attempt < max_attempts:
            print(f"Nouvel essai dans {wait} secondes")
            time.sleep(wait)
            wait *= 2

    raise RuntimeError("Overpass est indisponible après plusieurs essais.")


if os.path.exists(target_path):
    print("Collecte du jour déjà présente, téléchargement ignoré.")
else:
    raw_text = fetch_with_backoff()

    data = json.loads(raw_text)
    elements = data.get("elements", [])
    if len(elements) < 1000:
        raise ValueError(f"Seulement {len(elements)} éléments : réponse suspecte, rien n'est enregistré.")

    os.makedirs(target_dir, exist_ok=True)
    with open(target_path, "w", encoding="utf-8") as file:
        file.write(raw_text)
    print("Fichier enregistré :", target_path)

with open(target_path, encoding="utf-8") as file:
    elements = json.load(file)["elements"]

print("Éléments :", len(elements))
print("Avec coordonnées :", sum(1 for e in elements if "lat" in e or "center" in e))
for key in ["name", "brand", "operator", "addr:city", "opening_hours"]:
    filled = sum(1 for e in elements if key in e.get("tags", {}))
    print(f"Tag {key} : {filled} / {len(elements)}")